In [1]:
import pandas as pd
from sklearn.impute import KNNImputer

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
test["Transported"] = False
df = pd.concat([train,test],sort=False)
df.drop(['Name','PassengerId'],axis=1,inplace=True)
df.head()



,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [3]:
df.shape[0] == train.shape[0] + test.shape[0]

True

In [4]:
df.isna().sum()

HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
dtype: int64

In [5]:
df[['Deck','Num','Side']] = df['Cabin'].str.split('/',expand=True)
df = df.drop(columns='Cabin')
df['Deck'] = df['Deck'].fillna('U')
df['Num'] = df['Num'].fillna(-1)
df['Side'] = df['Side'].fillna('U')

In [6]:
df['Num'].value_counts()


Num
-1      299
82       34
56       28
4        28
31       27
       ... 
1848      1
1847      1
1846      1
1844      1
1890      1
Name: count, Length: 1895, dtype: int64

In [7]:
df['Deck'] = df["Deck"].map({'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'U':7,'T':8})
df['Side'] = df['Side'].map({'s':-1,'P':1,'U':2})


In [8]:
imput_list = ['Age','VIP','Num','CryoSleep','Side','Deck','RoomService','ShoppingMall','Spa','VRDeck','FoodCourt']
rest = list(set(df.columns)-set(imput_list))
df_rest = df[rest]
imp = KNNImputer(n_neighbors=5)
df_imputed = imp.fit_transform(df[imput_list])
df_imputed = pd.DataFrame(df_imputed,columns=imput_list)
df = pd.concat([df_rest.reset_index(drop=True),df_imputed.reset_index(drop=True)],axis=1)

In [9]:
df['HomePlanet'] = df['HomePlanet'].fillna('U')
df['Destination'] = df['Destination'].fillna('U')

In [12]:
bill_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
df['amt_spent'] = df[bill_cols].sum(axis=1)
df['std_amt_spent'] = df[bill_cols].std(axis=1)
df['mean_amt_spent'] = df[bill_cols].mean(axis=1)

df['3_high_cols'] = df['CryoSleep']+df['HomePlanet_Europa']+df['Destination_55 Cancri e']
df['3_low_cols'] = df['HomePlanet_Earth']+df['mean_amt_spent']+df['amt_spent']

In [11]:
cat_cols = ['HomePlanet','Destination']
for col in cat_cols:
    df = pd.concat([df,pd.get_dummies(df[col],prefix = col)],axis = 1)
df = df.drop(columns=cat_cols)


In [14]:
df.corr()['Transported'].sort_values(ascending=False)

Transported                  1.000000
CryoSleep                    0.324309
3_high_cols                  0.284117
HomePlanet_Europa            0.131977
Destination_55 Cancri e      0.083625
FoodCourt                    0.034792
HomePlanet_U                 0.006403
HomePlanet_Mars              0.005643
ShoppingMall                 0.004249
Destination_PSO J318.5-22    0.000760
Destination_U               -0.000554
Side                        -0.010828
VIP                         -0.018720
Num                         -0.035240
Age                         -0.050108
Destination_TRAPPIST-1e     -0.072731
Deck                        -0.086961
HomePlanet_Earth            -0.119644
std_amt_spent               -0.121208
mean_amt_spent              -0.140478
amt_spent                   -0.140478
3_low_cols                  -0.140501
VRDeck                      -0.142858
Spa                         -0.154805
RoomService                 -0.174912
Name: Transported, dtype: float64

In [15]:
train, df_test = df[:train.shape[0]], df[train.shape[0]:]
df_test = df_test.drop(columns='Transported')
train.shape,df_test.shape

((8693, 25), (4277, 24))

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [17]:
X = train.drop(columns='Transported')
Y = train['Transported']
X_train ,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=48)

In [18]:
models = {'LogisticRegression':LogisticRegression(),
          'DecisionTree':DecisionTreeClassifier(),
          'RandomForest':RandomForestClassifier(),
          'xgbboost':XGBClassifier(),
          'lightgbm':LGBMClassifier()
          }
names = []
for name, model in models.items():
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(Y_test,y_pred)
    names.append(name)

    
    print(f"{name:<20} | {acc:<10}")

c:\Users\Public\anaconda3\envs\analysis_env\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression   | 0.7837837837837838
DecisionTree         | 0.7372052903967797
RandomForest         | 0.7809085681426107
xgbboost             | 0.7935595169637722
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006087 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2709
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
lightgbm             | 0.7889591719378953


In [21]:
model = XGBClassifier()
model.fit(X,Y)
pred = model.predict(df_test)
df_dummy = pd.read_csv('test.csv')
final = pd.DataFrame()
final['PassengerId'] = df_dummy['PassengerId']
final['Transported'] = pred.astype(bool)
final.to_csv('submission.csv', index=False)